<a href="https://colab.research.google.com/github/yusuftiryaki/agac-goruntuleme/blob/main/AgacGoruntuleme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- COLAB ÇEVRE (ENVIRONMENT) KURULUM HÜCRESİ ---
# Bu hücreyi Notebook'un en tepesinde, Python kodlarından önce sadece 1 kez çalıştır.

# 1. GPU Kontrolü (Çıktıda Tesla T4 veya daha iyisini görmelisin)
!nvidia-smi

# 2. Sistem Araçları (COLMAP, FFmpeg, Sanal Ekran, ZBar)
!sudo apt-get update -y
!sudo apt-get install -y colmap ffmpeg xvfb libzbar0

# 3. Derleyici Araçları ve TinyCUDANN
# DİKKAT: PyTorch'u ellemiyoruz! Colab'ın güncel sürümünü kullanıyoruz.
# Bu adımın derlenmesi uzun sürer (yaklaşık 15-20 dk).

!pip install "/content/drive/MyDrive/OBA/tinycudann-2.0-cp312-cp312-linux_x86_64.whl

# 4. Nerfstudio Kurulumu
!pip install nerfstudio

# 5. Kendi Python iş hattımız için gereken diğer kütüphaneler (Firebase, OpenCV, ZBar, Open3D)
!pip install firebase-admin pyzbar opencv-python open3d numpy plyfile

In [ ]:
import os
import cv2
import numpy as np
import open3d as o3d
from pyzbar.pyzbar import decode
import firebase_admin
from firebase_admin import credentials, firestore, storage
import subprocess
import time
import glob
import urllib.parse
import json
from plyfile import PlyData

In [ ]:

def analyze_video_for_hybrid_plate(video_path):
    """
    Videoyu tarar:
    1. QR koddan karmaşık Ağaç ID'sini okur (Veri).
    2. ArUco Marker'dan piksel genişliğini ölçer (Geometri/Cetvel).
    """
    print(f"[DEBUG] Hibrit plaka analizi başlıyor: {video_path}")
    cap = cv2.VideoCapture(video_path)

    # ArUco ayarları (4x4 sözlüğü - tarla için en okunaklısı)
    aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    parameters = cv2.aruco.DetectorParameters()
    detector = cv2.aruco.ArucoDetector(aruco_dict, parameters)

    tree_id = None
    marker_pixel_width = None
    frame_count = 0

    # İlk 300 karede (yaklaşık ilk 10 saniye) plakayı bulmaya çalışıyoruz
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_count > 300:
            break

        # İşlemeyi hızlandırmak için kareyi siyah/beyaza çevir
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # ---------------------------------------------------------
        # 1. GÖREV: QR KOD TARAMASI (Ağaç ID'si)
        # ---------------------------------------------------------
        if tree_id is None:
            decoded_objects = decode(gray)
            for obj in decoded_objects:
                tree_id = obj.data.decode('utf-8')
                print(f"[DEBUG] Frame {frame_count}'te QR Kod Okundu! Ağaç ID: {tree_id}")
                break # İlk bulduğunu al ve döngüden çık

        # ---------------------------------------------------------
        # 2. GÖREV: ARUCO TARAMASI (Geometrik Cetvel)
        # ---------------------------------------------------------
        if marker_pixel_width is None:
            corners, ids, rejectedImgPoints = detector.detectMarkers(gray)
            if ids is not None:
                # İlk ArUco'nun köşeleri: [Sol Üst, Sağ Üst, Sağ Alt, Sol Alt]
                c = corners[0][0]
                # Sol Üst ile Sağ Üst köşe arasındaki piksel mesafesini hesapla
                marker_pixel_width = np.linalg.norm(c[0] - c[1])
                print(f"[DEBUG] Frame {frame_count}'te ArUco Marker Bulundu!")
                print(f"[DEBUG] ArUco Piksel Genişliği: {marker_pixel_width:.2f}")

        # ---------------------------------------------------------
        # ÇIKIŞ KOŞULU: İkisi de bulunduysa videoyu okumayı bırak (Zaman tasarrufu)
        # ---------------------------------------------------------
        if tree_id is not None and marker_pixel_width is not None:
            print(f"[DEBUG] Hibrit plaka başarıyla çözüldü. Video analizi durduruluyor.")
            break

        frame_count += 1

    cap.release()
    return tree_id, marker_pixel_width, frame_count

In [ ]:

def convert_ply_to_splat(ply_file_path, splat_file_path):
    """
    Nerfstudio'nun ürettiği ağır .ply dosyasını okur, matematiksel dönüşümleri
    (Sigmoid, Exp, SH_C0) yapar ve web için optimize edilmiş .splat (binary) formatına çevirir.
    """
    print(f"[DEBUG] Web optimizasyonu başlıyor. .ply -> .splat formatına sıkıştırılıyor...")

    # 1. PLY verisini oku
    plydata = PlyData.read(ply_file_path)
    vertex = plydata['vertex']

    num_pts = len(vertex['x'])

    # 2. Koordinatlar (X, Y, Z)
    x = np.array(vertex['x'])
    y = np.array(vertex['y'])
    z = np.array(vertex['z'])

    # 3. Renkler (Spherical Harmonics'ten RGB'ye çevrim)
    # Formül: RGB = (SH_DC * 0.28209) + 0.5
    SH_C0 = 0.28209479177387814
    if 'f_dc_0' in vertex.properties:
        r = np.clip((np.array(vertex['f_dc_0']) * SH_C0 + 0.5), 0.0, 1.0) * 255.0
        g = np.clip((np.array(vertex['f_dc_1']) * SH_C0 + 0.5), 0.0, 1.0) * 255.0
        b = np.clip((np.array(vertex['f_dc_2']) * SH_C0 + 0.5), 0.0, 1.0) * 255.0
    else:
        r, g, b = np.zeros(num_pts), np.zeros(num_pts), np.zeros(num_pts)

    # 4. Şeffaflık / Opacity (Sigmoid fonksiyonu uygulanır)
    if 'opacity' in vertex.properties:
        opacity_raw = np.array(vertex['opacity'])
        opacity = 1.0 / (1.0 + np.exp(-opacity_raw)) # Sigmoid
        a = opacity * 255.0
    else:
        a = np.ones(num_pts) * 255.0

    # 5. Boyutlar / Scale (Eksponansiyel fonksiyon uygulanır)
    if 'scale_0' in vertex.properties:
        sx = np.exp(np.array(vertex['scale_0']))
        sy = np.exp(np.array(vertex['scale_1']))
        sz = np.exp(np.array(vertex['scale_2']))
    else:
        sx, sy, sz = np.ones(num_pts), np.ones(num_pts), np.ones(num_pts)

    # 6. Dönüş / Rotation (Quaternion normalizasyonu)
    if 'rot_0' in vertex.properties:
        rw = np.array(vertex['rot_0'])
        rx = np.array(vertex['rot_1'])
        ry = np.array(vertex['rot_2'])
        rz = np.array(vertex['rot_3'])

        # Normalize et
        norms = np.sqrt(rw**2 + rx**2 + ry**2 + rz**2)
        rw, rx, ry, rz = rw/norms, rx/norms, ry/norms, rz/norms

        # 0-255 arasına map et: (val * 128) + 128
        qw = np.clip((rw * 128) + 128, 0, 255)
        qx = np.clip((rx * 128) + 128, 0, 255)
        qy = np.clip((ry * 128) + 128, 0, 255)
        qz = np.clip((rz * 128) + 128, 0, 255)
    else:
        qw, qx, qy, qz = np.ones(num_pts)*255, np.zeros(num_pts), np.zeros(num_pts), np.zeros(num_pts)

    # 7. BINARY SIKIŞTIRMA (32-byte struct)
    # Her bir nokta (splat) tam olarak 32 bayt yer kaplayacak şekilde paketlenir.
    dt = np.dtype([
        ('x', 'f4'), ('y', 'f4'), ('z', 'f4'),             # 12 byte (Koordinat)
        ('sx', 'f4'), ('sy', 'f4'), ('sz', 'f4'),          # 12 byte (Boyut)
        ('r', 'u1'), ('g', 'u1'), ('b', 'u1'), ('a', 'u1'),# 4 byte (Renk + Şeffaflık)
        ('qx', 'u1'), ('qy', 'u1'), ('qz', 'u1'), ('qw', 'u1') # 4 byte (Açı)
    ])

    splat_data = np.zeros(num_pts, dtype=dt)
    splat_data['x'], splat_data['y'], splat_data['z'] = x, y, z
    splat_data['sx'], splat_data['sy'], splat_data['sz'] = sx, sy, sz
    splat_data['r'], splat_data['g'], splat_data['b'], splat_data['a'] = r, g, b, a
    splat_data['qx'], splat_data['qy'], splat_data['qz'], splat_data['qw'] = qx, qy, qz, qw

    # Dosyayı kaydet
    with open(splat_file_path, "wb") as f:
        f.write(splat_data.tobytes())

    print(f"[SONUÇ] Mükemmel! Model başarıyla .splat formatına sıkıştırıldı: {splat_file_path}")
    return splat_file_path

In [ ]:

def calculate_fractal_dimension(pcd_crown):
    """
    Sadece yaprak ve dalları içeren taç kısmının (Nokta Bulutu)
    3D Kutu Sayma (Box-Counting) yöntemiyle fraktal skorunu hesaplar.
    """
    print("[DEBUG] Taç yapısının Fraktal Skoru hesaplanıyor...")

    # Noktaları numpy dizisine çevir
    points = np.asarray(pcd_crown.points)

    if len(points) == 0:
        return 0.0

    # 1. Noktaları Normalize Et (Tüm modeli 0 ile 1 arasında bir küpün içine sıkıştır)
    points_min = points.min(axis=0)
    points_max = points.max(axis=0)
    points_normalized = (points - points_min) / (points_max - points_min)

    # 2. Kutu boyutlarını (Ölçekleri) belirle
    # 2^1'den 2^8'e kadar logaritmik olarak artan ızgara (grid) çözünürlükleri
    scales = np.logspace(1, 8, num=8, endpoint=True, base=2)

    Ns = [] # Dolu kutu sayılarını tutacağımız liste

    for scale in scales:
        # Noktaların o anki ızgara (grid) içindeki koordinatlarını tam sayıya yuvarlayarak bul
        grid_indices = np.floor(points_normalized * scale).astype(int)

        # Benzersiz (içinde en az 1 nokta olan) kutuları say
        # axis=0 ile [x, y, z] koordinat üçlülerinin benzersiz olanlarını buluyoruz
        unique_boxes = len(np.unique(grid_indices, axis=0))
        Ns.append(unique_boxes)

    # 3. Log-Log Grafiğinin Eğimini Hesapla (Lineer Regresyon / Polyfit)
    # x ekseni: log(scale) -> Kutu sayısının tersi (1/epsilon)
    # y ekseni: log(Ns) -> Dolu kutu sayısı
    coeffs = np.polyfit(np.log(scales), np.log(Ns), 1)

    fractal_score = coeffs[0] # Doğrunun eğimi bize doğrudan Fraktal Boyutu (D_f) verir

    print(f"[SONUÇ] Ağacın Fraktal Skoru (D_f): {fractal_score:.3f}")

    return fractal_score

In [ ]:

def calculate_scale_factor(transforms_json_path, frame_count, pixel_width, real_width_m=0.15):
    """
    Kamera matematiğini kullanarak 3D model ile gerçek dünya arasındaki ölçek çarpanını bulur.
    """
    print("[DEBUG] Ölçek Çarpanı (Scale Factor) hesaplanıyor...")

    # 1. COLMAP'in oluşturduğu kamera verilerini oku
    with open(transforms_json_path, 'r') as f:
        data = json.load(f)

    # 2. Odak uzaklığını (Focal Length) al (COLMAP'in kusursuz hesapladığı değer)
    fl_x = data["fl_x"]

    # 3. GERÇEK DÜNYA MESAFESİ (Kamera -> Ağaç Gövdesi)
    real_distance_m = (fl_x * real_width_m) / pixel_width
    print(f"[DEBUG] Kameranın ağaca GERÇEK mesafesi: {real_distance_m:.2f} metre")

    # 4. SANAL DÜNYA MESAFESİ (COLMAP'in evrenindeki mesafe)
    # FFmpeg kareleri genelde frame_00001.jpg gibi kaydeder, formatı ayarlıyoruz
    # (frame_count + 1 yapıyoruz çünkü index 0'dan başlıyor, dosya isimleri 1'den)
    target_frame_name = f"frame_{frame_count + 1:05d}"

    virtual_distance = None
    for frame in data["frames"]:
        if target_frame_name in frame["file_path"]:
            # transform_matrix 4x4 bir matristir. X, Y, Z konumu sağ üsttedir.
            matrix = np.array(frame["transform_matrix"])
            camera_x = matrix[0, 3]
            camera_y = matrix[1, 3]
            camera_z = matrix[2, 3]

            # Kameranın merkeze (0,0,0) olan 3D uzaklığını hesapla (Öklid Mesafesi)
            virtual_distance = math.sqrt(camera_x**2 + camera_y**2 + camera_z**2)
            print(f"[DEBUG] Kameranın ağaca SANAL mesafesi: {virtual_distance:.4f} birim")
            break

    if virtual_distance is None:
        print("UYARI: ArUco bulunan kare transforms.json içinde bulunamadı! Varsayılan çarpan 1.0 kullanılıyor.")
        return 1.0

    # 5. ALTIN ÇARPAN (SCALE FACTOR)
    scale_factor = real_distance_m / virtual_distance
    print(f"[SONUÇ] Hesaplanan Ölçek Çarpanı: {scale_factor:.4f}")

    return scale_factor

In [ ]:
def measure_tree_metrics(ply_file_path, scale_factor):
    """
    3D nokta bulutunu alır, ArUco referansına göre metrik (metre) sisteme çevirir,
    toprağı bulur ve ağacın boyunu/hacmini hesaplar.
    """
    print(f"[DEBUG] 3D Model analiz ediliyor: {ply_file_path}")

    # 1. Nokta Bulutunu Yükle
    pcd = o3d.io.read_point_cloud(ply_file_path)

    # 2. Gürültü Temizleme (Havada asılı kalan hatalı pikselleri sil)
    # nb_neighbors: Her noktanın etrafındaki komşu sayısı, std_ratio: Standart sapma toleransı
    cl, ind = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
    pcd = pcd.select_by_index(ind)

    # 3. ÖLÇEKLEME (Scaling) - Cetveli Vurmak
    # Katsayı = Gerçek Boyut (metre) / COLMAP'in Sanal Boyutu
    print(f"[DEBUG] Uygulanan Ölçek Çarpanı: {scale_factor:.4f}")

    # Tüm modeli ölçeklendir (Artık model metre cinsinden)
    pcd.scale(scale_factor, center=pcd.get_center())

    # 4. ZEMİN TESPİTİ (Toprağı Bulmak)
    # Noktaların Z eksenini (yukarı/aşağı) al. En alttaki %1'lik dilimi zemin kabul et.
    # (%1 kullanıyoruz çünkü tek tük aşağı sarkan gürültü noktalarına aldanmak istemeyiz)
    points = np.asarray(pcd.points)
    z_coords = points[:, 2] # 2. indeks Z eksenidir
    ground_z = np.percentile(z_coords, 1.0)

    # Tüm ağacı aşağı/yukarı kaydırarak toprağı tam Z = 0 noktasına oturt
    translation_vector = np.array([0, 0, -ground_z])
    pcd.translate(translation_vector)

    # 5. AĞAÇ BOYUNU HESAPLAMA
    # Toprak artık Z=0 olduğuna göre, en yüksek Z noktası bize boyu verir
    points_shifted = np.asarray(pcd.points)
    tree_height_m = np.max(points_shifted[:, 2])

    # 6. TAÇ HACMİ HESAPLAMA (Convex Hull)
    # Gövdeyi hesaba katmamak için topraktan itibaren ilk 50 cm'yi (0.5m) kırpıyoruz
    crown_indices = np.where(points_shifted[:, 2] > 0.5)[0]
    crown_pcd = pcd.select_by_index(crown_indices)

    # Kalan yapraklı kısmın etrafını sanal bir zarla sar (Convex Hull) ve hacmini ölç
    hull, _ = crown_pcd.compute_convex_hull()
    crown_volume_m3 = hull.get_volume()

    # 7. GÖVDE ÇAPI HESAPLAMA (Trunk Diameter)
    # Fıstık ağaçları için topraktan 30 cm ile 50 cm (0.3m - 0.5m) arasındaki net gövde dilimini alıyoruz
    trunk_indices = np.where((points_shifted[:, 2] > 0.3) & (points_shifted[:, 2] < 0.5))[0]
    trunk_pcd = pcd.select_by_index(trunk_indices)

    # Bu dilimdeki noktaların sadece X ve Y koordinatlarını al (Kuş bakışı 2D görünüme geç)
    trunk_points_2d = np.asarray(trunk_pcd.points)[:, :2]

    if len(trunk_points_2d) > 10: # Eğer o yükseklikte yeterli gövde noktası varsa
        # 1. Gövdenin tam merkez noktasını (ortalama X ve Y) bul
        center_2d = np.mean(trunk_points_2d, axis=0)

        # 2. Merkezden gövdenin dış kabuğundaki tüm noktalara olan mesafeleri ölç
        distances = np.linalg.norm(trunk_points_2d - center_2d, axis=1)

        # 3. Yarıçapı bul. (Kamera gürültüsünden/çöplerden etkilenmemek için median -ortanca- değeri alıyoruz)
        radius_m = np.median(distances)

        # 4. Yarıçapı 2 ile çarpıp çapa ulaş, ardından santimetreye çevir
        trunk_diameter_cm = (radius_m * 2) * 100

        print(f"[SONUÇ] Gövde Çapı (30-50cm arası): {trunk_diameter_cm:.1f} cm")
    else:
        trunk_diameter_cm = 0.0
        print("[UYARI] Gövde çapı ölçülemedi (O yükseklikte yeterli nokta bulunamadı).")

    # 8. Fraktal Boyur hesaplama
    crown_indices = np.where(points_shifted[:, 2] > 0.5)[0]
    crown_pcd = pcd.select_by_index(crown_indices)
    fractal_score = calculate_fractal_dimension(crown_pcd)

    print(f"[SONUÇ] Ağaç Boyu: {tree_height_m:.2f} Metre")
    print(f"[SONUÇ] Taç Hacmi: {crown_volume_m3:.2f} Metreküp")
    print(f"[SONUÇ] Ağaç Gövde Çapı: {trunk_diameter_cm:.2f} Santimetre")
    print(f"[SONUÇ] Ağaç Fraktal Boyutu: {fractal_score:.2f} Skalar")

    return tree_height_m, crown_volume_m3,trunk_diameter_cm, fractal_score, ground_z, pcd

In [ ]:
# --- 1. FIREBASE BAĞLANTISI ---
# Firebase konsolundan indirdiğiniz serviceAccountKey.json dosyasını Colab'a yükleyin
CREDENTIALS_PATH = "/content/drive/MyDrive/OBA/firebase.json"
STORAGE_BUCKET = "studio-166104040-52130.firebasestorage.app" # Kendi bucket isminizle değiştirin

if not firebase_admin._apps:
    cred = credentials.Certificate(CREDENTIALS_PATH)
    firebase_admin.initialize_app(cred, {
        'storageBucket': STORAGE_BUCKET
    })

db = firestore.client()
bucket = storage.bucket()

# Sistem Sabitleri
QR_GERCEK_BOYUT_METRE = 0.15  # 15 cm x 15 cm QR kod kullandığımızı varsayıyoruz
WORKSPACE_DIR = "/content/pistachio_workspace"
os.makedirs(WORKSPACE_DIR, exist_ok=True)


In [ ]:
def run_colmap_processing(video_path, doc_id, colmap_dir):
    """FFmpeg kullanarak videodan kareler çıkarır ve COLMAP ile kamera pozisyonlarını hesaplar."""
    print(f"[{doc_id}] Adım 1/3: FFmpeg ve COLMAP çalışıyor (Bu adım videonun uzunluğuna göre 5-15 dk sürer)...")

    colmap_db_path = os.path.join(colmap_dir, "colmap", "database.db")
    if os.path.exists(colmap_db_path):
        print(f"[DEBUG] COLMAP veritabanı {colmap_db_path} zaten mevcut. Adım 1 atlanıyor.")
        return

    try:
        cmd = [
            "xvfb-run", "-a", # Sanal ekran tetikleyici eklendi!
            "ns-process-data", "video",
            "--data", video_path,
            "--output-dir", colmap_dir,
            "--num-frames-target", "60",         # Test için 60 kareye indirdik (hızlı bitsin)
            "--matching-method", "sequential",   # Videolar için işlemciyi kurtaran sihirli ayar
            "--verbose"
        ]
        print(f"[DEBUG] Running command: {' '.join(cmd)}")

        # Qt platform ve OpenGL hatalarını çözmek için ortam değişkenlerini ayarla
        env = os.environ.copy()
        env["QT_QPA_PLATFORM"] = "offscreen"
        env["LIBGL_ALWAYS_SOFTWARE"] = "1" # OpenGL'i yazılımsal renderlemeye zorla

        result = subprocess.run(cmd, check=True, capture_output=True, text=True, env=env)
        print("ns-process-data stdout:\n", result.stdout)
        print("ns-process-data stderr:\n", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"ns-process-data command failed. Exit code: {e.returncode}")
        print(f"ns-process-data stdout on error:\n{e.stdout}")
        print(f"ns-process-data stderr on error:\n{e.stderr}")
        raise Exception(f"ns-process-data komutu başarısız oldu: {e}")

In [ ]:
def train_splatfacto_model(doc_id, colmap_dir):
    """3D Gaussian Splatting (Splatfacto) modelini eğitir."""
    print(f"[{doc_id}] Adım 2/3: 3DGS modeli eğitiliyor (GPU tam kapasite devrede, 15-25 dk sürer)....")

    # Eğitilmiş modelin config.yml dosyasının varlığını kontrol et
    colmap_folder_name = os.path.basename(colmap_dir)
    config_search_path = f"{WORKSPACE_DIR}/{doc_id}/outputs/{colmap_folder_name}/splatfacto/*/config.yml"
    existing_configs = glob.glob(config_search_path)

    if existing_configs:
        print(f"[DEBUG] Eğitilmiş model {existing_configs[0]} zaten mevcut. Adım 2 atlanıyor.")
        return

    try:
      custom_env = os.environ.copy()
      custom_env["TORCH_COMPILE_DISABLE"] = "1"
      custom_env["MAX_JOBS"] = "2"
      abs_colmap_dir = os.path.abspath(colmap_dir)
      splat_result = subprocess.run([
          "ns-train", "splatfacto",
          "--data", abs_colmap_dir,
          "--max-num-iterations", "7000",
          "--vis", "tensorboard" # ÇÖZÜM 1: Canlı izleme arayüzünü tamamen kapatır, port çökmesini önler.
      ], check=True, capture_output=True, text=True, env=custom_env, cwd=f"{WORKSPACE_DIR}/{doc_id}")

      print("ns-train stdout:\n", splat_result.stdout)
      print("ns-train stderr:\n", splat_result.stderr)

    except subprocess.CalledProcessError as e:
      print(f"ns-train command failed. Exit code: {e.returncode}")
      print(f"ns-train stdout on error:\n{e.stdout}")
      print(f"ns-train stderr on error:\n{e.stderr}")
      raise Exception(f"ns-train komutu başarısız oldu: {e}")

In [ ]:

def run_3d_reconstruction_pipeline(video_path, doc_id):
    print(f"[{doc_id}] GERÇEK 3D ÜRETİM BAŞLIYOR...")

    # Her ağaç için izole bir çalışma alanı yarat
    base_dir = f"{WORKSPACE_DIR}/{doc_id}"
    colmap_dir = f"{base_dir}/colmap_data"
    export_dir = f"{base_dir}/export"
    os.makedirs(colmap_dir, exist_ok=True)
    os.makedirs(export_dir, exist_ok=True)


    # ADIM 1: FFmpeg ve COLMAP (Kareleri çıkar ve kamera pozisyonlarını hesapla)
    run_colmap_processing(video_path, doc_id, colmap_dir)

    # ADIM 2: 3D Gaussian Splatting (Splatfacto) Eğitimi
    train_splatfacto_model(doc_id=doc_id, colmap_dir=colmap_dir)

    # ADIM 3: Eğitilen Modeli .PLY formatında dışa aktarma
    print("Adım 3/3: Model .ply formatında dışa aktarılıyor...")

    # Nerfstudio eğitilen modelin config dosyasını dinamik bir klasöre atar. Onu bulmamız lazım.
    # Yol formatı: outputs/<colmap_klasörü_adı>/splatfacto/<tarih_saat>/config.yml
    colmap_folder_name = os.path.basename(colmap_dir)
    config_search_path = f"{base_dir}/outputs/{colmap_folder_name}/splatfacto/*/config.yml"

    try:
        config_path = glob.glob(config_search_path)[0]
    except IndexError:
        raise Exception("Eğitim tamamlandı ama config.yml bulunamadı. Model çöküp üretilememiş olabilir.")

    # ns-export komutu gaussian splatting modelini ölçüm yapabileceğimiz .ply dosyasına çevirir.
    subprocess.run(
        [
            "ns-export", "gaussian-splat",
            "--load-config", config_path,
            "--output-dir", export_dir
        ], check=True,cwd=f"{base_dir}"
    )

    # ns-export komutu varsayılan olarak dosyayı 'splat.ply' adıyla kaydeder
    final_ply_path = os.path.join(export_dir, "splat.ply")

    print(f"[{doc_id}] GERÇEK 3D ÜRETİM TAMAMLANDI. Çıktı: {final_ply_path}")

    # Bu ply dosyası artık Open3D'ye (önceki koddaki scale_and_measure_tree fonksiyonuna) gidiyor.
    return final_ply_path

In [ ]:

def upload_metrics_and_update_db(local_image_path, splat_model_path, doc_id, height, diameter, volume, fractal_score, scale_factor, ground_z):
    """
    Oluşturulan 3D ve 2D metrik görüntüsünü Firebase Storage'a yükler,
    herkese açık (public) URL'sini alır ve tüm metrikleri Firestore'a kaydeder.
    """
    print(f"[DEBUG] 3D model Firebase'e yükleniyor: {splat_model_path}")
    cloud_model_path = f"models/{doc_id}_model.splat"
    out_blob = bucket.blob(cloud_model_path)
    out_blob.upload_from_filename(splat_model_path)
    out_blob.make_public()
    model_public_url = out_blob.public_url

    print(f"[DEBUG] 2D Metrik görüntüsü Firebase'e yükleniyor: {local_image_path}")

    # 1. STORAGE'A YÜKLEME VE URL ALMA
    bucket = storage.bucket() # Firebase storage bucket'ınızı çağırır

    # Storage içindeki dosya yolunu ve adını belirle (Örn: trees/AGAC_42/metrics_view.jpg)
    destination_blob_name = f"trees/{doc_id}/metrics_view.jpg"
    blob = bucket.blob(destination_blob_name)

    # Dosyayı Colab'dan Firebase'e fırlat
    blob.upload_from_filename(local_image_path)

    # Web uygulamasından şifresiz/doğrudan görüntülenebilmesi için public yapıyoruz
    blob.make_public()
    image_public_url = blob.public_url

    print(f"[SONUÇ] Resim başarıyla yüklendi! Public URL: {image_public_url}")

    # 2. FIRESTORE VERİTABANINI GÜNCELLEME
    print(f"[DEBUG] Firestore veritabanı güncelleniyor: {doc_id}")
    db = firestore.client()

    # Hem resmi hem de hesapladığımız tüm o harika metrikleri veritabanına yazıyoruz
    tree_data = {
        "status": "completed",
        "measurements": {
            "height_m": round(height, 2),
            "diameter_cm": round(diameter, 1),
            "volume_m3": round(volume, 2),
            "fractal_score": round(fractal_score, 3)
        },
        "transform": {
            "scale": scale_factor,
            "z_offset": float(ground_z) # Numpy float'ını standart float'a çeviriyoruz
        },
        'model_url': model_public_url,
        "metrics_image_url": image_public_url,
        "last_updated": firestore.SERVER_TIMESTAMP
    }

    # Belgeyi (dokümanı) birleştirerek (merge=True) güncelle
    # (Böylece PWA'dan gelen diğer veriler -konum, notlar vb.- silinmez)
    db.collection("trees").document(doc_id).set(tree_data, merge=True)

    print(f"[SONUÇ] {doc_id} numaralı ağacın tüm dijital ikiz verileri Firebase'e işlendi!")

    return image_public_url, model_public_url

In [ ]:
def is_video_h264_mp4(video_path):
    """Videoun H264 codec ve yuv420p piksel formatında olup olmadığını kontrol eder."""
    try:
        cmd = [
            "ffprobe",
            "-v", "error",
            "-select_streams", "v:0",
            "-show_entries", "stream=codec_name,pix_fmt",
            "-of", "json",
            video_path
        ]
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        info = json.loads(result.stdout)

        if "streams" in info and len(info["streams"]) > 0:
            video_stream = info["streams"][0]
            if video_stream.get("codec_name") == "h264" and video_stream.get("pix_fmt") == "yuv420p":
                return True
        return False
    except (subprocess.CalledProcessError, json.JSONDecodeError) as e:
        print(f"[WARNING] ffprobe failed for {video_path}: {e}")
        return False # ffprobe başarısız olursa, formatın istenen gibi olmadığını varsayalım

In [ ]:

def create_2d_metrics_view(pcd, doc_id, height, diameter, volume):
    """
    3D nokta bulutunu kullanarak yan yana (Kuş Bakışı ve Yandan Görünüm)
    iki planlı bir 2D metrik tablosu (Blueprint) oluşturur.
    """
    print(f"[DEBUG] 2D Kapsamlı Metrik Görüntüsü oluşturuluyor: {doc_id}")

    # 1. Nokta Bulutunu Yükle ve Noktaları Al
    points = np.asarray(pcd.points)

    # 2. AYARLAR VE TUVAL BOYUTLARI
    ppm = 150  # Piksel/Metre (1 metre = 150 piksel olacak şekilde ölçekle)
    margin = 1.0  # Kenar boşluğu (metre)

    # Modelin sınırlarını (Bounding Box) bul
    x_min, x_max = np.min(points[:, 0]), np.max(points[:, 0])
    y_min, y_max = np.min(points[:, 1]), np.max(points[:, 1])
    z_min, z_max = np.min(points[:, 2]), np.max(points[:, 2]) # z_min 0'a çok yakındır

    # Alt Tuvallerin Piksel Genişlik ve Yüksekliklerini Hesapla
    # Sol Tuval (Kuş Bakışı - XY)
    top_w = int((x_max - x_min + 2*margin) * ppm)
    top_h = int((y_max - y_min + 2*margin) * ppm)

    # Sağ Tuval (Yandan Görünüm - XZ)
    side_w = int((x_max - x_min + 2*margin) * ppm)
    side_h = int((z_max - z_min + 2*margin) * ppm)

    # Ana Tuval Boyutları (Yan yana iki resim)
    canvas_h = max(top_h, side_h)
    canvas_w = top_w + side_w

    # Beyaz bir ana tuval oluştur (Mimari arka plan)
    metrics_img = np.ones((canvas_h, canvas_w, 3), dtype=np.uint8) * 255

    # Çizgiler için renkler (BGR formatı)
    c_tree = (34, 139, 34)    # Orman Yeşili (Ağaç pikselleri)
    c_measure = (255, 100, 0) # Turuncu (Mimari ölçü okları)
    c_text = (50, 50, 50)     # Koyu Gri (Metinler)
    c_ground = (40, 70, 130)  # Kahverengi Tonu (Toprak çizgisi)

    # -------------------------------------------------------------------
    # 3. SOL TARAF: KUŞ BAKIŞI (TOP VIEW) - Taç Hacmi
    # -------------------------------------------------------------------
    # X ve Y koordinatlarını piksellere çevir
    px_top = ((points[:, 0] - x_min + margin) * ppm).astype(int)
    py_top = ((points[:, 1] - y_min + margin) * ppm).astype(int)

    # Numpy vektörizasyonu ile noktaları tuvale tek hamlede bas
    metrics_img[py_top, px_top] = c_tree

    # Ağacın tam merkezini bul
    cx_top = int((0 - x_min + margin) * ppm)
    cy_top = int((0 - y_min + margin) * ppm)

    # Merkez işaretçisi ve Hacim Bilgisi
    cv2.drawMarker(metrics_img, (cx_top, cy_top), c_measure, markerType=cv2.MARKER_CROSS, markerSize=20, thickness=2)
    cv2.putText(metrics_img, "KUS BAKISI (XY)", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, c_text, 2)
    cv2.putText(metrics_img, f"Tac Hacmi: {volume:.2f} m3", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, c_measure, 2)

    # Ortaya ayıran dikey bir çizgi çek
    cv2.line(metrics_img, (top_w, 0), (top_w, canvas_h), (200, 200, 200), 2)

    # -------------------------------------------------------------------
    # 4. SAĞ TARAF: YANDAN GÖRÜNÜM (SIDE VIEW) - Boy ve Çap
    # -------------------------------------------------------------------
    # X ve Z koordinatlarını piksellere çevir
    px_side = ((points[:, 0] - x_min + margin) * ppm + top_w).astype(int)
    # Görüntülerde Y ekseni aşağı doğru artar, bu yüzden Z eksenini ters çeviriyoruz
    py_side = (canvas_h - (points[:, 2] - z_min + margin) * ppm).astype(int)

    metrics_img[py_side, px_side] = c_tree

    # Zemin (Toprak) Çizgisi
    ground_y = int(canvas_h - (0 - z_min + margin) * ppm)
    cv2.line(metrics_img, (top_w + 20, ground_y), (canvas_w - 20, ground_y), c_ground, 3)
    cv2.putText(metrics_img, "Zemin (Z=0)", (top_w + 20, ground_y + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, c_ground, 2)

    # Merkez X (Sağ tuvaldeki X ekseni ortası)
    cx_side = int((0 - x_min + margin) * ppm + top_w)

    # --- BOY ÖLÇÜSÜ (Dikey Ok) ---
    top_y = int(canvas_h - (height - z_min + margin) * ppm)
    offset_x = 80 # Ağacın biraz yanından çizelim ki karışmasın
    cv2.arrowedLine(metrics_img, (cx_side + offset_x, ground_y), (cx_side + offset_x, top_y), c_measure, 2, tipLength=0.05)
    cv2.arrowedLine(metrics_img, (cx_side + offset_x, top_y), (cx_side + offset_x, ground_y), c_measure, 2, tipLength=0.05)
    cv2.putText(metrics_img, f"Boy: {height:.2f} m", (cx_side + offset_x + 10, int((ground_y + top_y)/2)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, c_measure, 2)

    # --- GÖVDE ÇAPI ÖLÇÜSÜ (Yatay Ok) ---
    # Çapı topraktan yaklaşık 40 cm (0.4m) yukarıdan çiziyoruz
    trunk_y = int(canvas_h - (0.4 - z_min + margin) * ppm)
    radius_px = int((diameter / 200) * ppm) # Çapı yarıçapa ve cm'den metreye çevir

    cv2.arrowedLine(metrics_img, (cx_side - radius_px - 30, trunk_y), (cx_side - radius_px, trunk_y), c_measure, 2)
    cv2.putText(metrics_img, f"Cap: {diameter:.1f} cm", (cx_side - radius_px - 140, trunk_y + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, c_measure, 2)

    cv2.putText(metrics_img, "YANDAN GORUNUM (XZ)", (top_w + 20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, c_text, 2)

    # -------------------------------------------------------------------
    # 5. KAYDET
    # -------------------------------------------------------------------
    # Çıktı klasörünün var olduğundan emin ol
    os.makedirs(f"{WORKSPACE_DIR}/{doc_id}/outputs", exist_ok=True)

    metrics_view_path = f"{WORKSPACE_DIR}/{doc_id}/outputs/{doc_id}_metrics.jpg"
    cv2.imwrite(metrics_view_path, metrics_img)
    print(f"[SONUÇ] Mükemmel 2D Mimari Plan kaydedildi: {metrics_view_path}")

    return metrics_view_path

In [ ]:
def process_pending_trees():
    """Firestore'da 'bekliyor' (pending) durumundaki ağaçları bulur ve işler."""
    print("Firebase kontrol ediliyor...")
    pending_docs = db.collection('trees').where('status', '==', 'pending').stream()

    for doc in pending_docs:
        tree_data = doc.to_dict()
        doc_id = doc.id
        video_url_in_storage = tree_data.get('video_url') # PWA'nın storage'a kaydettiği yol

        print(f"Yeni iş bulundu: {doc_id}. İşlem başlatılıyor...")
        print(f"Depolama yolu: {video_url_in_storage}")

        # Durumu güncelliyoruz ki başka bir worker (eğer yatay büyürseniz) bunu almasın
        db.collection('trees').document(doc_id).update({'status': 'processing'})

        try:
            if not video_url_in_storage:
                raise Exception("Video yolu Firebase'de bulunamadı veya boş!")

            # URL'den blob adını çıkar
            parsed_url = urllib.parse.urlparse(video_url_in_storage)
            path_segments = parsed_url.path.split('/')
            # Blob adı '/o/' kısmından sonraki ve '?alt=media' kısmından önceki kısımdır.
            # Genellikle 7. segmenttir (v0, b, bucket_name, o, videos, filename)
            blob_name_encoded = '/'.join(path_segments[path_segments.index('o') + 1:])
            blob_name = urllib.parse.unquote(blob_name_encoded) # URL-decode the blob name

            # 1. Videoyu İndir
            local_video_path = os.path.join(WORKSPACE_DIR, f"{doc_id}.mp4")
            blob = bucket.blob(blob_name)
            blob.download_to_filename(local_video_path)
            print("Video indirildi.")

            # YENİ ADIM: OpenCV uyumluluğunu artırmak ve gereksiz yeniden kodlamayı önlemek için kontrol et
            reencoded_video_path = os.path.join(WORKSPACE_DIR, f"{doc_id}_reencoded.mp4")

            if is_video_h264_mp4(local_video_path):
                print(f"[DEBUG] Video {local_video_path} zaten H264 MP4 (yuv420p) formatında. Yeniden kodlama atlanıyor.")
                video_to_analyze = local_video_path
            else:
                print(f"[DEBUG] Videoyu ffmpeg kullanarak {local_video_path} konumundan {reencoded_video_path} konumuna yeniden kodluyor...")
                try:
                    subprocess.run([
                        "ffmpeg", "-y", "-i", local_video_path,
                        "-vcodec", "libx264", "-pix_fmt", "yuv420p",
                        "-preset", "medium", "-crf", "23",
                        reencoded_video_path
                    ], check=True, capture_output=True, text=True)
                    print("[DEBUG] Video yeniden kodlama başarılı.")
                    video_to_analyze = reencoded_video_path
                except subprocess.CalledProcessError as e:
                    print(f"[ERROR] Video yeniden kodlama başarısız oldu: {e.stderr}")
                    video_to_analyze = local_video_path # Yeniden kodlama başarısız olursa orijinal videoyu kullan
                except FileNotFoundError:
                    print("[ERROR] ffmpeg komutu bulunamadı. Yeniden kodlama atlanıyor.")
                    video_to_analyze = local_video_path
            print(f"[DEBUG] Analiz edilecek video yolu: {video_to_analyze}")

            # 2. QR Kod Analizi ve Ağaç ID Tespiti
            qr_data, marker_pixel_width, detected_frame_count = analyze_video_for_hybrid_plate(video_to_analyze)

            if qr_data is None or marker_pixel_width is None:
                print("HATA: QR Kod veya ArUco Marker videonun başında net olarak bulunamadı!")
                # Burada Firestore'a "başarısız - plaka okunamadı" durumu basıp bir sonraki videoya geçebilirsiniz.
            else:
                print(f"Başarılı! Ağaç ID: {qr_data} | Referans Piksel: {marker_pixel_width}")


            # 3. 3D Model Üretimi (COLMAP + 3DGS)
            # Not: Bu fonksiyon arka planda bash scriptlerini tetikler.
            # PoC için üretilmiş bir .ply dosyası döndürdüğünü varsayıyoruz.
            raw_model_path = run_3d_reconstruction_pipeline(video_to_analyze, doc_id)



            # 4. Ölçeklendirme ve Hacim/Boy Ölçümü (Open3D)
            # transforms.json dosyasının yolu (ns-process-data bunu oluşturur)
            transforms_path = f"/content/pistachio_workspace/{doc_id}/colmap_data/transforms.json"

            if marker_pixel_width is not None:
              # Daha önce hibrit plakayı okurken frame_count'u ve pixel_width'i değişkende tuttuğumuzu varsayıyoruz
              # Çarpanı Hesapla
              scale_factor = calculate_scale_factor(transforms_path, detected_frame_count, marker_pixel_width)
              height, volume,trunk_diameter_cm, fractal_score, ground_z, pcd  = measure_tree_metrics(
                  raw_model_path,
                  scale_factor
              )
              print(f"Ölçümler Tamamlandı: Boy: {height:.2f}m, Hacim: {volume:.2f}m3 , Gövde Çapı: {trunk_diameter_cm:.2f}cm , Fraktal Boyut: {fractal_score:.2f}")

              # 5. 3D model 2D metrik imajı oluşturma
              local_image_path = create_2d_metrics_view(pcd, doc_id, height, trunk_diameter_cm, volume)

            # Web için .splat dönüşümünü yap
            splat_model_path = raw_model_path.replace(".ply", ".splat")
            convert_ply_to_splat(raw_model_path, splat_model_path)

            # 6. Firebase'e Geri Yükleme ve Güncelleme
            upload_metrics_and_update_db(
                local_image_path=local_image_path,
                splat_model_path=splat_model_path,
                doc_id=doc_id,
                height=height,
                diameter=trunk_diameter_cm,
                volume=volume,
                fractal_score=fractal_score,
                scale_factor=scale_factor,
                ground_z=ground_z
            )


        except Exception as e:
            print(f"HATA: {doc_id} işlenirken hata oluştu: {str(e)}")
            db.collection('trees').document(doc_id).update({'status': 'pending', 'error': str(e)})

In [ ]:
# --- CERRAHİ YAMA: PyTorch 3.12 Dynamo Hatasını Fiziksel Olarak Kapat ---
patch_file_path = "/usr/local/lib/python3.12/dist-packages/nerfstudio/utils/misc.py"
if os.path.exists(patch_file_path):
    with open(patch_file_path, "r") as f:
        content = f.read()

    # Hata veren kodu, boş dönen bir lambda fonksiyonuyla değiştir
    old_code = "return torch.compile(*args, **kwargs)"
    new_code = "return lambda x: x"

    if old_code in content:
        content = content.replace(old_code, new_code)
        with open(patch_file_path, "w") as f:
            f.write(content)
        print("[SİSTEM] Nerfstudio Dynamo hatası başarıyla yamalandı!")
# -----------------------------------------------------------------------

# Nerfstudio'nun model yükleme dosyasının yolunu buluyoruz
file_path = '/usr/local/lib/python3.12/dist-packages/nerfstudio/utils/eval_utils.py'

# Dosyayı okuyoruz
with open(file_path, 'r') as file:
    content = file.read()

# PyTorch 2.6'nın güvenlik duvarını aşacak parametreyi koda zorla enjekte ediyoruz
eski_kod = 'torch.load(load_path, map_location="cpu")'
yeni_kod = 'torch.load(load_path, map_location="cpu", weights_only=False)'
content = content.replace(eski_kod, yeni_kod)

# Yamalı dosyayı geri kaydediyoruz
with open(file_path, 'w') as file:
    file.write(content)

print("Nerfstudio, PyTorch 2.6 güvenlik duvarına karşı başarıyla yamalandı!")

In [27]:
!rm -r /content/pistachio_workspace/mhehdjn1fBIAbRLylWdR.mp4

In [ ]:
# --- İŞÇİYİ BAŞLAT ---
if __name__ == "__main__":
    # Colab defterinde bu hücreyi çalıştırdığınızda bekleyen işleri tarar
    process_pending_trees()

Firebase kontrol ediliyor...
Yeni iş bulundu: mhehdjn1fBIAbRLylWdR. İşlem başlatılıyor...
Depolama yolu: https://firebasestorage.googleapis.com/v0/b/studio-166104040-52130.firebasestorage.app/o/videos%2Fgemini1.mp4?alt=media&token=307a2dc7-3277-41aa-944f-e090ba792653


/usr/local/lib/python3.12/dist-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


Video indirildi.
[DEBUG] Video /content/pistachio_workspace/mhehdjn1fBIAbRLylWdR.mp4 zaten H264 MP4 (yuv420p) formatında. Yeniden kodlama atlanıyor.
[DEBUG] Analiz edilecek video yolu: /content/pistachio_workspace/mhehdjn1fBIAbRLylWdR.mp4
[DEBUG] Hibrit plaka analizi başlıyor: /content/pistachio_workspace/mhehdjn1fBIAbRLylWdR.mp4
[DEBUG] Frame 21'te ArUco Marker Bulundu!
[DEBUG] ArUco Piksel Genişliği: 19.31
HATA: QR Kod veya ArUco Marker videonun başında net olarak bulunamadı!
[mhehdjn1fBIAbRLylWdR] GERÇEK 3D ÜRETİM BAŞLIYOR...
[mhehdjn1fBIAbRLylWdR] Adım 1/3: FFmpeg ve COLMAP çalışıyor (Bu adım videonun uzunluğuna göre 5-15 dk sürer)...
[DEBUG] Running command: xvfb-run -a ns-process-data video --data /content/pistachio_workspace/mhehdjn1fBIAbRLylWdR.mp4 --output-dir /content/pistachio_workspace/mhehdjn1fBIAbRLylWdR/colmap_data --num-frames-target 150
